# 02b — Silver & Gold (Standalone Execution)

**Use this notebook** when you want to demo the medallion flow interactively without standing up the Lakeflow pipeline. It produces the same tables as `02_medallion_pipeline` using `CREATE OR REPLACE TABLE` instead of `dlt.table()`.

> The Lakeflow SDP version (02_medallion_pipeline) is the production artifact. This notebook exists for fast, interactive demo walkthroughs with visible cell output.

In [0]:
UC_CATALOG = "serverless_stable_qr9if1_catalog"
UC_SCHEMA = "mccain_cold_chain_sju"
spark.sql(f"USE {UC_CATALOG}.{UC_SCHEMA}")

In [0]:
%sql
-- Silver: cleaned + enriched telemetry with shipment/product context
CREATE OR REPLACE TABLE serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.silver_reefer_telemetry AS
SELECT
    rt.trailer_id, rt.shipment_id, rt.ts,
    rt.set_point_c, rt.supply_air_c, rt.return_air_c,
    rt.compressor_on, rt.fuel_level_pct, rt.alarm_code,
    s.sku, s.carrier_id, s.dest_dc_id, s.pallets, s.ship_ts, s.arrival_ts,
    pm.product_family, pm.target_temp_c, pm.upper_tolerance_c,
    pm.lower_tolerance_c, pm.unit_value_usd,
    (rt.supply_air_c > pm.upper_tolerance_c) AS is_above_tolerance
FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.bronze_reefer_telemetry rt
JOIN serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.shipments       s  USING (shipment_id)
JOIN serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.product_master  pm ON s.sku = pm.sku
WHERE rt.supply_air_c IS NOT NULL
  AND rt.ts BETWEEN s.ship_ts - INTERVAL 1 HOUR
               AND s.arrival_ts + INTERVAL 1 HOUR

In [0]:
from pyspark.sql.functions import col

# --- EXECUTION EVIDENCE: Data Quality checks ---
print("="*60)
print("SILVER DATA QUALITY VALIDATION")
print("="*60)

dq = {}
t = spark.table(f"{UC_CATALOG}.{UC_SCHEMA}.silver_reefer_telemetry")
dq["rows_total"]          = t.count()
dq["rows_null_supply"]    = t.filter(col("supply_air_c").isNull()).count()
dq["rows_absurd_temp"]    = t.filter((col("supply_air_c") < -50) | (col("supply_air_c") > 20)).count()
dq["rows_orphan_shipment"] = (t.join(spark.table(f"{UC_CATALOG}.{UC_SCHEMA}.shipments"),
                                     "shipment_id", "left_anti").count())
for k, v in dq.items():
    status = "✅" if v == 0 or k == "rows_total" else "❌"
    print(f"  {status} {k:<22} {v:>8,}")

assert dq["rows_null_supply"] == 0,    "Null supply temp after silver — STOP"
assert dq["rows_absurd_temp"] == 0,    "Physically impossible temps — STOP"
assert dq["rows_orphan_shipment"] == 0, "Telemetry without matching shipment — STOP"
print(f"\n✅ Silver DQ: all {dq['rows_total']:,} rows clean")

In [0]:
%sql
-- Silver: one row per shipment with temp stats, door events, value
CREATE OR REPLACE TABLE serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.silver_shipment_profile AS
WITH door_open_mid AS (
    SELECT d.shipment_id, sum(CASE WHEN d.event = 'door_open' THEN 1 ELSE 0 END) AS mid_transit_door_opens
    FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.bronze_door_events d
    JOIN serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.shipments s USING (shipment_id)
    WHERE d.ts BETWEEN s.ship_ts + INTERVAL 10 MINUTE AND s.arrival_ts - INTERVAL 10 MINUTE
    GROUP BY d.shipment_id
)
SELECT
    s.shipment_id, s.trailer_id, s.carrier_id, s.dest_dc_id, s.sku, s.pallets,
    s.ship_ts, s.arrival_ts,
    (unix_timestamp(s.arrival_ts) - unix_timestamp(s.ship_ts))/3600.0 AS transit_hours,
    pm.upper_tolerance_c, pm.unit_value_usd,
    round(min(rt.supply_air_c), 2)                                          AS min_supply_c,
    round(max(rt.supply_air_c), 2)                                          AS max_supply_c,
    round(avg(rt.supply_air_c), 2)                                          AS avg_supply_c,
    sum(CASE WHEN rt.is_above_tolerance THEN 1 ELSE 0 END)                  AS minutes_above_tolerance,
    sum(CASE WHEN rt.alarm_code IS NOT NULL THEN 1 ELSE 0 END)              AS alarm_minutes,
    max(CASE WHEN rt.alarm_code IS NOT NULL THEN rt.alarm_code END)         AS top_alarm,
    coalesce(dm.mid_transit_door_opens, 0)                                  AS mid_transit_door_opens,
    (max(rt.supply_air_c) > pm.upper_tolerance_c)                           AS had_excursion,
    s.pallets * pm.unit_value_usd * 40                                      AS shipment_value_usd
FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.silver_reefer_telemetry rt
JOIN serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.shipments       s  USING (shipment_id)
JOIN serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.product_master  pm ON s.sku = pm.sku
LEFT JOIN door_open_mid dm ON dm.shipment_id = s.shipment_id
GROUP BY s.shipment_id, s.trailer_id, s.carrier_id, s.dest_dc_id, s.sku, s.pallets,
         s.ship_ts, s.arrival_ts, pm.upper_tolerance_c, pm.unit_value_usd, dm.mid_transit_door_opens

In [0]:
%sql
-- EXECUTION EVIDENCE: excursion vs clean shipment summary
SELECT had_excursion, count(*) AS shipments,
       round(avg(max_supply_c), 2) AS avg_peak_c,
       round(sum(shipment_value_usd), 0) AS total_value_usd
FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.silver_shipment_profile
GROUP BY had_excursion

In [0]:
%sql
-- Gold: one row per contiguous excursion event with severity + dollars at risk
CREATE OR REPLACE TABLE serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.gold_excursion_events AS
WITH flagged AS (
    SELECT
        shipment_id, trailer_id, carrier_id, dest_dc_id, sku, product_family,
        ts, supply_air_c, upper_tolerance_c, unit_value_usd, pallets,
        is_above_tolerance,
        sum(CASE WHEN NOT is_above_tolerance THEN 1 ELSE 0 END)
            OVER (PARTITION BY shipment_id ORDER BY ts ROWS UNBOUNDED PRECEDING) AS grp
    FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.silver_reefer_telemetry
),
windows AS (
    SELECT
        shipment_id, trailer_id, carrier_id, dest_dc_id, sku, product_family, pallets, unit_value_usd,
        grp,
        min(ts)                                 AS start_ts,
        max(ts)                                 AS end_ts,
        round(max(supply_air_c), 2)             AS peak_supply_c,
        round(max(supply_air_c) - avg(upper_tolerance_c), 2) AS peak_over_tolerance_c,
        count(*)                                AS duration_min
    FROM flagged
    WHERE is_above_tolerance
    GROUP BY shipment_id, trailer_id, carrier_id, dest_dc_id, sku, product_family, pallets, unit_value_usd, grp
)
SELECT
    concat(shipment_id, '-EX', lpad(cast(grp AS string), 3, '0')) AS excursion_id,
    shipment_id, trailer_id, carrier_id, dest_dc_id,
    sku, product_family, pallets,
    start_ts, end_ts, duration_min,
    peak_supply_c, peak_over_tolerance_c,
    CASE
      WHEN duration_min >= 60 AND peak_over_tolerance_c >= 3 THEN 'HIGH'
      WHEN duration_min >= 20 OR peak_over_tolerance_c >= 2   THEN 'MED'
      ELSE 'LOW'
    END AS severity,
    round(
      pallets * unit_value_usd * 40 *
      CASE WHEN duration_min >= 60 AND peak_over_tolerance_c >= 3 THEN 1.0
           WHEN duration_min >= 20 OR peak_over_tolerance_c >= 2 THEN 0.35
           ELSE 0.05 END,
    0) AS dollars_at_risk
FROM windows
WHERE duration_min >= 5

In [0]:
%sql
-- EXECUTION EVIDENCE: excursion severity breakdown with dollars at risk
SELECT severity, count(*) AS events,
       sum(dollars_at_risk) AS total_dollars_at_risk,
       round(avg(duration_min)) AS avg_minutes
FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.gold_excursion_events
GROUP BY severity
ORDER BY CASE severity WHEN 'HIGH' THEN 1 WHEN 'MED' THEN 2 ELSE 3 END

In [0]:
%sql
-- EXECUTION EVIDENCE: The "punch list" for ops — top 10 worst events
SELECT excursion_id, trailer_id, carrier_id, dest_dc_id, sku,
       start_ts, duration_min, peak_supply_c, peak_over_tolerance_c,
       severity, dollars_at_risk
FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.gold_excursion_events
ORDER BY dollars_at_risk DESC
LIMIT 10